# PCR Lab Data -- ETL
Dane: https://zenodo.org/records/11617408

Parsowanie plików YAML do Parquet.  
Uruchomić raz -- kolejne notebooki czytają z `data/processed/`.

In [25]:
%pip install -q pandas numpy pyyaml pyarrow tqdm 2>/dev/null

Note: you may need to restart the kernel to use updated packages.


In [19]:
%%bash
mkdir -p ../data/raw
curl -L --fail -o ../data/raw/zenodo.zip https://zenodo.org/api/records/11617408/files-archive
unzip -oq ../data/raw/zenodo.zip -d ../data/raw/zenodo_11617408

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0

100 30.3M    0 30.3M    0     0  2563k      0 --:--:--  0:00:12 --:--:-- 4429k--:--:--  0:00:01 --:--:--  285k3k--:-- 2096k473k72k


In [26]:
import yaml
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm.auto import tqdm

ROOT = Path('..').resolve()
DATA_DIR = ROOT / 'data' / 'raw'
PROCESSED_DIR = ROOT / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

yaml_files = sorted(
    p for p in DATA_DIR.rglob('*.xes.yaml')
    if '__MACOSX' not in p.parts and not p.name.startswith('._')
)
print(f'YAML files: {len(yaml_files)}')

YAML files: 6339


In [27]:
%%time

RESULT_MAP = {'N': 'NEGATIVE', 'P': 'POSITIVE', 'NEGATIVE': 'NEGATIVE', 'POSITIVE': 'POSITIVE'}

def normalize_result(val):
    if isinstance(val, dict):
        val = val.get('result', None)
    if isinstance(val, str):
        return RESULT_MAP.get(val.strip().upper(), val)
    return None

def extract_result_from_data(data_list):
    result, ct = None, None
    if not isinstance(data_list, list):
        return result, ct
    for item in data_list:
        if not isinstance(item, dict):
            continue
        if item.get('name') == 'state' and isinstance(item.get('value'), dict):
            state = item['value']
            result = normalize_result(state.get('result'))
            ct = state.get('ct')
        elif item.get('name') == 'result':
            result = normalize_result(item.get('value'))
        elif item.get('name') == 'ct':
            ct = item.get('value')
    return result, ct


def parse_yaml_file(filepath):
    events = []
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            docs = list(yaml.safe_load_all(f))

        instance_uuid = filepath.stem.replace('.xes', '')

        header = docs[0] if docs else {}
        trace = header.get('log', {}).get('trace', {}) if isinstance(header, dict) else {}
        case_name = trace.get('cpee:name', '')

        for doc in docs[1:]:
            if not isinstance(doc, dict) or 'event' not in doc:
                continue
            ev = doc['event']

            ts = ev.get('time:timestamp', None)
            if hasattr(ts, 'isoformat'):
                ts = ts.isoformat()
            elif not isinstance(ts, str):
                ts = None

            result, ct = extract_result_from_data(ev.get('data'))

            events.append({
                'case_id': ev.get('concept:instance'),
                'instance_uuid': instance_uuid,
                'case_name': case_name,
                'activity': ev.get('concept:name'),
                'endpoint': ev.get('concept:endpoint', ''),
                'lifecycle': ev.get('lifecycle:transition'),
                'cpee_lifecycle': ev.get('cpee:lifecycle:transition'),
                'cpee_activity_id': ev.get('id:id', ''),
                'cpee_state': ev.get('cpee:state'),
                'timestamp': ts,
                'result': result,
                'ct': ct,
            })
    except Exception:
        pass
    return events


all_events = []
for filepath in tqdm(yaml_files, desc='Parsing'):
    all_events.extend(parse_yaml_file(filepath))

df_all = pd.DataFrame(all_events)
df_all['timestamp'] = pd.to_datetime(df_all['timestamp'], utc=True, errors='coerce')
df_all['ct'] = pd.to_numeric(df_all['ct'], errors='coerce')
df_all['endpoint'] = df_all['endpoint'].fillna('').astype(str)

print(f'\nTotal events: {len(df_all):,} from {df_all["instance_uuid"].nunique():,} cases')

Parsing: 100%|██████████| 6339/6339 [19:03<00:00,  5.54it/s]   



Total events: 317,905 from 6,339 cases
CPU times: user 18min 49s, sys: 15.6 s, total: 19min 4s
Wall time: 19min 12s


In [28]:
INVALID_NAMES = {None, '__INVALID__', 'external'}
BUSINESS_LIFECYCLES = {'start', 'complete'}

df_biz = df_all[
    (~df_all['activity'].isin(INVALID_NAMES)) &
    (df_all['activity'].notna()) &
    (df_all['lifecycle'].isin(BUSINESS_LIFECYCLES))
].copy()
df_biz = df_biz.sort_values(['instance_uuid', 'timestamp']).reset_index(drop=True)

print(f'Business events: {len(df_biz):,}')
print(f'Cases: {df_biz["instance_uuid"].nunique():,}')

Business events: 134,099
Cases: 6,339


In [29]:
case_stats = df_biz.groupby('instance_uuid').agg(
    n_events=('activity', 'count'),
    n_activities=('activity', 'nunique'),
    first_ts=('timestamp', 'min'),
    last_ts=('timestamp', 'max'),
).reset_index()
case_stats['duration_min'] = (case_stats['last_ts'] - case_stats['first_ts']).dt.total_seconds() / 60

name_map = df_biz.drop_duplicates('instance_uuid').set_index('instance_uuid')['case_name']
case_stats['case_name'] = case_stats['instance_uuid'].map(name_map)

def classify_process(name):
    if not isinstance(name, str) or name == '':
        return 'unknown'
    if name.startswith('Sample:'):
        return 'sample'
    if name.startswith('Wellplate:'):
        return 'wellplate'
    if name in ('Lab Plain Instance', 'Lab Finish Watcher'):
        return name.lower().replace(' ', '_')
    return 'other'

case_stats['process_type'] = case_stats['case_name'].apply(classify_process)

results_raw = df_all[df_all['result'].notna()].drop_duplicates(subset='instance_uuid', keep='last')
result_map = results_raw.set_index('instance_uuid')['result'].to_dict()
case_stats['pcr_result'] = case_stats['instance_uuid'].map(result_map)

ct_raw = df_all[(df_all['ct'].notna()) & (df_all['ct'] > 0)].drop_duplicates(subset='instance_uuid', keep='last')
ct_map = ct_raw.set_index('instance_uuid')['ct'].to_dict()
case_stats['ct'] = case_stats['instance_uuid'].map(ct_map)

print(f'Cases: {len(case_stats)}')
print(f'With PCR result: {case_stats["pcr_result"].notna().sum()}')
print(f'With Ct > 0: {case_stats["ct"].notna().sum()}')
print()
print('Process types:')
print(case_stats['process_type'].value_counts().to_string())

Cases: 6339
With PCR result: 5993
With Ct > 0: 2827

Process types:
process_type
sample                6166
wellplate              165
lab_plain_instance       4
lab_finish_watcher       4


In [30]:
df_biz_save = df_biz.copy()
df_biz_save['result'] = df_biz_save['result'].astype(str).where(df_biz_save['result'].notna(), None)
df_biz_save['ct'] = pd.to_numeric(df_biz_save['ct'], errors='coerce')
df_biz_save['endpoint'] = df_biz_save['endpoint'].astype(str)

out_events = PROCESSED_DIR / 'pcr_events_biz.parquet'
df_biz_save.to_parquet(out_events, index=False)
print(f'Events: {out_events.relative_to(ROOT)} ({out_events.stat().st_size / 1024:.1f} KB)')

case_stats_save = case_stats.copy()
case_stats_save['pcr_result'] = case_stats_save['pcr_result'].astype(str).where(
    case_stats_save['pcr_result'].notna(), None)

out_cases = PROCESSED_DIR / 'pcr_cases.parquet'
case_stats_save.to_parquet(out_cases, index=False)
print(f'Cases:  {out_cases.relative_to(ROOT)} ({out_cases.stat().st_size / 1024:.1f} KB)')

print(f'\nColumns events: {list(df_biz_save.columns)}')
print(f'Columns cases:  {list(case_stats_save.columns)}')

Events: data/processed/pcr_events_biz.parquet (1603.9 KB)
Cases:  data/processed/pcr_cases.parquet (592.8 KB)

Columns events: ['case_id', 'instance_uuid', 'case_name', 'activity', 'endpoint', 'lifecycle', 'cpee_lifecycle', 'cpee_activity_id', 'cpee_state', 'timestamp', 'result', 'ct']
Columns cases:  ['instance_uuid', 'n_events', 'n_activities', 'first_ts', 'last_ts', 'duration_min', 'case_name', 'process_type', 'pcr_result', 'ct']
